# 🌍 Geografische Analyse: Region, District & Branch Office

In dieser Analyse untersuchen wir die geografische Struktur der Kunden:
- Anzahl und Liste der Regionen (1. Ebene)
- Anzahl und Verteilung der Districts (2. Ebene)
- Anzahl der Branch Offices pro Region
- Kundenverteilung je Region, District und Branch Office

In [3]:
import pandas as pd
import os

# Pfad an Notebook-Struktur angepasst
data_path = os.path.join("..", "data", "dataset_wuerth.csv")

# Daten laden
df = pd.read_csv(data_path)

print(f"✅ Datensatz geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")

✅ Datensatz geladen: 29493 Zeilen, 42 Spalten


In [24]:
import os
import pandas as pd

# Pfad relativ zum Notebook (liegt in notebooks/, Daten in data/)
data_path = os.path.join("..", "data", "dataset_wuerth.csv")
df = pd.read_csv(data_path)

# --- Basischecks ---
regions = sorted(df["region"].dropna().unique().tolist())
n_regions = len(regions)
n_districts_total = df["district"].nunique(dropna=True)

# Districts pro Region
districts_per_region = (
    df.groupby("region")["district"]
      .nunique(dropna=True)
      .rename("districts_count")
      .reset_index()
      .sort_values("region")
)

# Branch Offices pro Region (NaN herausfiltern)
branch_per_region = (
    df.dropna(subset=["branch_office"])
      .groupby("region")["branch_office"]
      .nunique()
      .rename("branch_office_count")
      .reset_index()
      .sort_values("region")
)

# Kunden pro Region (zur Einordnung)
customers_per_region = (
    df["region"].value_counts().sort_index().rename("customers")
    .reset_index().rename(columns={"index":"region"})
)

# --- Text-Report zusammenbauen ---
lines = []
lines.append("🧭 Geografische Struktur – Kurzreport")
lines.append("-" * 60)
lines.append(f"Anzahl Regionen: {n_regions}   ({', '.join(map(str, regions))})")
lines.append(f"Anzahl Districts gesamt: {n_districts_total}")

# Districts je Region
lines.append("\nDistricts je Region:")
for _, r in districts_per_region.iterrows():
    lines.append(f"  Region {int(r['region'])}: {int(r['districts_count'])} Districts")

# Branch Offices je Region
lines.append("\nBranch Offices je Region:")
for _, r in branch_per_region.iterrows():
    lines.append(f"  Region {int(r['region'])}: {int(r['branch_office_count'])} Branch Offices")

# Kunden je Region (Top-Info)
lines.append("\nKunden je Region (Anzahl):")
for _, r in customers_per_region.iterrows():
    lines.append(f"  Region {int(r['region'])}: {int(r['customers'])} Kunden")

report_text = "\n".join(lines)
print(report_text)
#display(districts_per_region)
#display(branch_per_region)
#display(customers_per_region)

🧭 Geografische Struktur – Kurzreport
------------------------------------------------------------
Anzahl Regionen: 8   (11, 12, 13, 14, 15, 16, 17, 18)
Anzahl Districts gesamt: 175

Districts je Region:
  Region 11: 24 Districts
  Region 12: 20 Districts
  Region 13: 23 Districts
  Region 14: 18 Districts
  Region 15: 23 Districts
  Region 16: 24 Districts
  Region 17: 24 Districts
  Region 18: 19 Districts

Branch Offices je Region:
  Region 11: 24 Branch Offices
  Region 12: 30 Branch Offices
  Region 13: 48 Branch Offices
  Region 14: 23 Branch Offices
  Region 15: 23 Branch Offices
  Region 16: 18 Branch Offices
  Region 17: 24 Branch Offices
  Region 18: 15 Branch Offices

Kunden je Region (Anzahl):
  Region 11: 3717 Kunden
  Region 12: 3479 Kunden
  Region 13: 4536 Kunden
  Region 14: 3053 Kunden
  Region 15: 4227 Kunden
  Region 16: 3110 Kunden
  Region 17: 4172 Kunden
  Region 18: 3199 Kunden


In [11]:
import os
import numpy as np
import pandas as pd

# === Daten laden ===
data_path = os.path.join("..", "data", "dataset_wuerth.csv")
df = pd.read_csv(data_path)

# Sicherstellen, dass das ORSY-Flag 0/1 ist
df["flag_new_orsyshelf"] = (df["flag_new_orsyshelf"] > 0).astype(int)

# === Kanäle & Gesamterlös (ohne 'potential') ===
rev_cols = ["rev_salesrep", "rev_branch_office", "rev_ebusiness", "rev_internal_staff", "rev_others"]

# Gesamtumsatz aus Komponenten (robuster als 'sales', falls Abweichungen)
df["total_revenue"] = df[rev_cols].sum(axis=1)

# Shares je Kanal (gegen total_revenue, damit die Summe ≈ 1 ist)
for c in rev_cols:
    df[f"{c}_share"] = np.where(df["total_revenue"] > 0, df[c] / df["total_revenue"], np.nan)

# === Schnellchecks (nur zur Info) ===
print("Datensatz:", df.shape)
print("ORSY-Anteil gesamt: {:.1f}%".format(df["flag_new_orsyshelf"].mean() * 100))
diff_sales = (df["sales"] - df["total_revenue"]).abs().mean()
print("Ø Abweichung zwischen 'sales' und Summe der Kanäle:", round(diff_sales, 2))

# === Gruppierung: Region × ORSY ===
region_summary = (
    df.groupby(["region", "flag_new_orsyshelf"])
      .agg(
          customers=("cust_id", "count"),
          avg_sales=("sales", "mean"),
          avg_orders=("orders_count", "mean"),
          avg_emp=("emp_count", "mean"),
          share_salesrep=("rev_salesrep_share", "mean"),
          share_branch=("rev_branch_office_share", "mean"),
          share_ebusiness=("rev_ebusiness_share", "mean"),
          share_internal=("rev_internal_staff_share", "mean"),
          share_others=("rev_others_share", "mean"),
      )
      .reset_index()
)

region_summary["flag_new_orsyshelf"] = region_summary["flag_new_orsyshelf"].map({0: "Nicht-ORSY", 1: "ORSY"})
display(region_summary.head(20))

# === Gruppierung: Region × District × ORSY (Drilldown) ===
district_summary = (
    df.groupby(["region", "district", "flag_new_orsyshelf"])
      .agg(
          customers=("cust_id", "count"),
          avg_sales=("sales", "mean"),
          avg_orders=("orders_count", "mean"),
          avg_emp=("emp_count", "mean"),
          share_salesrep=("rev_salesrep_share", "mean"),
          share_branch=("rev_branch_office_share", "mean"),
          share_ebusiness=("rev_ebusiness_share", "mean"),
          share_internal=("rev_internal_staff_share", "mean"),
          share_others=("rev_others_share", "mean"),
      )
      .reset_index()
)

district_summary["flag_new_orsyshelf"] = district_summary["flag_new_orsyshelf"].map({0: "Nicht-ORSY", 1: "ORSY"})
display(district_summary.head(20))

# === Struktur-Überblick je Region (Anzahl Districts & Branch Offices) ===
struct_summary = (
    df.assign(branch_office=df["branch_office"].astype(str))
      .dropna(subset=["branch_office"])
      .groupby("region")
      .agg(
          customers=("cust_id", "count"),
          districts_count=("district", "nunique"),
          branch_office_count=("branch_office", "nunique")
      )
      .reset_index()
      .sort_values("region")
)
display(struct_summary)

# === Exporte (für spätere Nutzung / Visualisierung) ===
out_dir = os.path.join("..", "data")
os.makedirs(out_dir, exist_ok=True)
region_summary.to_csv(os.path.join(out_dir, "region_channel_summary.csv"), index=False)
district_summary.to_csv(os.path.join(out_dir, "district_channel_summary.csv"), index=False)
struct_summary.to_csv(os.path.join(out_dir, "region_structure_counts.csv"), index=False)
print("✅ Exporte gespeichert unter ../data/:",
      "region_channel_summary.csv, district_channel_summary.csv, region_structure_counts.csv", sep="\n- ")

Datensatz: (29493, 48)
ORSY-Anteil gesamt: 9.4%
Ø Abweichung zwischen 'sales' und Summe der Kanäle: 0.0


,region,flag_new_orsyshelf,customers,avg_sales,avg_orders,avg_emp,share_salesrep,share_branch,share_ebusiness,share_internal,share_others
0,11,Nicht-ORSY,3387,2822.820520,15.803071,4.156185,0.225557,0.584230,0.130154,0.060583,-0.000525
1,11,ORSY,330,22189.842970,94.572727,7.103030,0.396646,0.313697,0.234101,0.056451,-0.000894
2,12,Nicht-ORSY,3181,2714.129154,15.368438,3.979881,0.227881,0.620709,0.108472,0.045600,-0.002663
3,12,ORSY,298,21002.933591,87.050336,7.208054,0.351315,0.401502,0.197256,0.052509,-0.002582
4,13,Nicht-ORSY,4127,2884.721737,15.965108,4.517567,0.186098,0.641826,0.119061,0.054150,-0.001135
5,13,ORSY,409,20555.963472,88.440098,7.591687,0.358264,0.382627,0.203995,0.057600,-0.002486
6,14,Nicht-ORSY,2760,2470.947638,14.884058,3.928261,0.254132,0.566058,0.111593,0.069839,-0.001623
7,14,ORSY,293,19250.709215,83.389078,6.784983,0.421395,0.273084,0.236644,0.070500,-0.001623
8,15,Nicht-ORSY,3856,2628.028662,14.690093,4.158195,0.232490,0.579657,0.131720,0.059982,-0.003849
9,15,ORSY,371,22088.986954,84.334232,6.948787,0.418711,0.334868,0.198253,0.049383,-0.001215


,region,district,flag_new_orsyshelf,customers,avg_sales,avg_orders,avg_emp,share_salesrep,share_branch,share_ebusiness,share_internal,share_others
0,11,60015,Nicht-ORSY,112,4376.335000,19.651786,5.473214,0.253103,0.436115,0.181618,0.129447,-0.000283
1,11,60015,ORSY,10,29585.062000,118.400000,7.300000,0.197005,0.377334,0.303744,0.122222,-0.000305
2,11,60779,Nicht-ORSY,148,2751.152905,15.797297,5.587838,0.247243,0.544705,0.140710,0.067496,-0.000154
3,11,60779,ORSY,11,22895.797273,98.636364,6.454545,0.461822,0.407664,0.125325,0.006284,-0.001094
4,11,60796,Nicht-ORSY,153,2790.666209,17.823529,2.856209,0.281440,0.558231,0.098521,0.062534,-0.000726
5,11,60796,ORSY,24,23921.642500,106.291667,5.958333,0.319986,0.304080,0.270647,0.106317,-0.001030
6,11,61070,Nicht-ORSY,157,3404.433949,21.388535,3.031847,0.209488,0.687392,0.063621,0.040811,-0.001313
7,11,61070,ORSY,17,22644.241176,88.588235,5.941176,0.296477,0.444925,0.200980,0.058390,-0.000773
8,11,61218,Nicht-ORSY,65,1426.477538,9.246154,2.784615,0.337033,0.567708,0.068343,0.027947,-0.001031
9,11,61218,ORSY,10,20546.163000,119.200000,5.200000,0.627125,0.039731,0.320533,0.012611,0.000000


,region,customers,districts_count,branch_office_count
0,11,3717,24,25
1,12,3479,20,31
2,13,4536,23,49
3,14,3053,18,24
4,15,4227,23,24
5,16,3110,24,19
6,17,4172,24,25
7,18,3199,19,16


✅ Exporte gespeichert unter ../data/:
- region_channel_summary.csv, district_channel_summary.csv, region_structure_counts.csv


In [22]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots

# ======================
# Daten laden & vorbereiten
# ======================
DATA_PATH = os.path.join("..", "data", "dataset_wuerth.csv")
df = pd.read_csv(DATA_PATH)

# ORSY-Flag als 0/1 und Label
df["orsy_flag"] = (df["flag_new_orsyshelf"] > 0).astype(int)
df["orsy_str"]  = df["orsy_flag"].map({0:"Nicht-ORSY", 1:"ORSY"})

# Optionale Korrektur: negative "Others" sind Korrekturbuchungen -> auf 0 kappen (analytisch neutral)
df["rev_others"] = df["rev_others"].clip(lower=0)

# ======================
# Klassen (Bins) definieren
# ======================
# Mitarbeiter: feiner am unteren Rand, gröber bei großen Firmen
emp_bins   = [-0.1, 1, 5, 10, 20, 50, 100, 200, np.inf]
emp_labels = ["0–1", "2–5", "6–10", "11–20", "21–50", "51–100", "101–200", "200+"]

# Orders: ähnlich staffeln
ord_bins   = [-0.1, 1, 5, 10, 20, 50, 100, 200, np.inf]
ord_labels = ["0–1", "2–5", "6–10", "11–20", "21–50", "51–100", "101–200", "200+"]

df["emp_bin"]  = pd.cut(df["emp_count"], bins=emp_bins, labels=emp_labels, include_lowest=True, right=True)
df["ord_bin"]  = pd.cut(df["orders_count"], bins=ord_bins, labels=ord_labels, include_lowest=True, right=True)

# Nur gültige Klassen behalten
df = df.dropna(subset=["emp_bin","ord_bin"])

# ======================
# Aggregation: Raster (emp_bin × ord_bin) je ORSY-Gruppe
# ======================
agg = (
    df.groupby(["orsy_str","emp_bin","ord_bin"], observed=True)
      .agg(
          customers=("cust_id","count"),
          sales_sum=("sales","sum")
      )
      .reset_index()
)

# Pivot zu 2D-Matrizen (Zeilen = emp_bin, Spalten = ord_bin)
def make_matrix(sub, value_col):
    mat = (
        sub.pivot_table(index="emp_bin", columns="ord_bin", values=value_col, fill_value=0)
          .reindex(index=emp_labels, columns=ord_labels)  # feste Reihenfolge
    )
    return mat

mat_non_count  = make_matrix(agg[agg["orsy_str"]=="Nicht-ORSY"], "customers")
mat_orsi_count = make_matrix(agg[agg["orsy_str"]=="ORSY"],       "customers")
mat_non_sales  = make_matrix(agg[agg["orsy_str"]=="Nicht-ORSY"], "sales_sum")
mat_orsi_sales = make_matrix(agg[agg["orsy_str"]=="ORSY"],       "sales_sum")

# ======================
# 1) Heatmaps – Kundenanzahl (nebeneinander)
# ======================
fig_counts = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Nicht-ORSY – Kundenanzahl", "ORSY – Kundenanzahl"),
    horizontal_spacing=0.08
)

fig_counts.add_trace(
    px.imshow(
        mat_non_count, origin="lower", color_continuous_scale="Viridis",
        labels=dict(x="Bestellungen (Binned)", y="Mitarbeiter (Binned)", color="Kunden")
    ).data[0],
    row=1, col=1
)

fig_counts.add_trace(
    px.imshow(
        mat_orsi_count, origin="lower", color_continuous_scale="Viridis",
        labels=dict(x="Bestellungen (Binned)", y="Mitarbeiter (Binned)", color="Kunden")
    ).data[0],
    row=1, col=2
)

fig_counts.update_xaxes(title_text="Bestellungen (Binned)", row=1, col=1)
fig_counts.update_yaxes(title_text="Mitarbeiter (Binned)",  row=1, col=1)
fig_counts.update_xaxes(title_text="Bestellungen (Binned)", row=1, col=2)
fig_counts.update_yaxes(title_text="Mitarbeiter (Binned)",  row=1, col=2)
fig_counts.update_layout(height=600, width=1200, coloraxis_colorbar=dict(title="Kunden"))

fig_counts.show()

# ======================
# 2) Heatmaps – Umsatzsumme € (nebeneinander)
# ======================
# Für bessere Lesbarkeit € in Tausendern anzeigen
mat_non_sales_k = mat_non_sales / 1_000
mat_orsi_sales_k = mat_orsi_sales / 1_000

fig_sales = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Nicht-ORSY – Umsatzsumme (Tsd. €)", "ORSY – Umsatzsumme (Tsd. €)"),
    horizontal_spacing=0.08
)

fig_sales.add_trace(
    px.imshow(
        mat_non_sales_k, origin="lower", color_continuous_scale="Blues",
        labels=dict(x="Bestellungen (Binned)", y="Mitarbeiter (Binned)", color="Tsd. €")
    ).data[0],
    row=1, col=1
)

fig_sales.add_trace(
    px.imshow(
        mat_orsi_sales_k, origin="lower", color_continuous_scale="Blues",
        labels=dict(x="Bestellungen (Binned)", y="Mitarbeiter (Binned)", color="Tsd. €")
    ).data[0],
    row=1, col=2
)

fig_sales.update_xaxes(title_text="Bestellungen (Binned)", row=1, col=1)
fig_sales.update_yaxes(title_text="Mitarbeiter (Binned)",  row=1, col=1)
fig_sales.update_xaxes(title_text="Bestellungen (Binned)", row=1, col=2)
fig_sales.update_yaxes(title_text="Mitarbeiter (Binned)",  row=1, col=2)
fig_sales.update_layout(height=600, width=1200, coloraxis_colorbar=dict(title="Tsd. €"))

fig_sales.show()

# ======================
# (Optional) Top-Zellen als Tabelle: wo steckt „Masse“?
# ======================
top_cells = (
    agg.sort_values(["orsy_str","sales_sum"], ascending=[True, False])
       .groupby("orsy_str").head(10)
       .assign(sales_sum_k=lambda d: (d["sales_sum"]/1_000).round(1))
       .drop(columns=["sales_sum"])
)
display(top_cells)

/var/folders/7w/3vw94kq12mbbs075sr0n34zr0000gp/T/ipykernel_41554/1661378417.py:52: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

/var/folders/7w/3vw94kq12mbbs075sr0n34zr0000gp/T/ipykernel_41554/1661378417.py:52: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

/var/folders/7w/3vw94kq12mbbs075sr0n34zr0000gp/T/ipykernel_41554/1661378417.py:52: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

/var/folders/7w/3vw94kq12mbbs075sr0n34zr0000gp/T/ipykernel_41554/1661378417.py:52: FutureWarning:

The default value of o

,orsy_str,emp_bin,ord_bin,customers,sales_sum_k
12,Nicht-ORSY,2–5,21–50,2258,12479.9
13,Nicht-ORSY,2–5,51–100,720,8127.1
11,Nicht-ORSY,2–5,11–20,2550,6491.6
21,Nicht-ORSY,6–10,51–100,300,4226.0
20,Nicht-ORSY,6–10,21–50,596,3837.5
10,Nicht-ORSY,2–5,6–10,2631,3492.5
14,Nicht-ORSY,2–5,101–200,156,3492.3
4,Nicht-ORSY,0–1,21–50,632,3157.0
9,Nicht-ORSY,2–5,2–5,4656,2532.3
22,Nicht-ORSY,6–10,101–200,98,2426.5
